# Dataset and descriptive statistics II

The goal of this notebook is to explore the AIDev dataset. Here it will download the dataset and run a few exploratory queries to give us a better understanding of the variables available for the research.

**AIDev v5** dataset ([`hao-li/AIDev`](https://huggingface.co/datasets/hao-li/AIDev)):

Main variables:
1. Agentic PRs per month by agent
2. Repos by month of first agentic PR
3. Hours to merge / hours to close per agent (box = IQR, whiskers = p10–p90)

`all_pull_request.parquet` is 6.7 GB. It is downloaded once into the Hugging Face cache
(`~/.cache/huggingface`); only the light columns are then streamed into a slim local copy
(`data/prs_light.parquet`), so re-running the notebook is fast and memory stays low.

In [1]:
! pip install pathlib matplotlib pandas pyarrow huggingface_hub


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

HF_REPO = "hao-li/AIDev"
HF_REVISION = "v5"  # pin dataset version - cutoff 2026-03-31)

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

LIGHT_PATH = DATA_DIR / "prs_light.parquet"
PR_COLUMNS = ["agent", "user", "state", "created_at", "closed_at", "merged_at", "repo_url"]

/Users/vagnersilva/Documents/masters/semester-3/Observational Studies/tu-graz-observational-studies/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download the PR table from Hugging Face

In [ ]:
def to_utc_naive(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, format="ISO8601", utc=True).dt.tz_localize(None)

if LIGHT_PATH.exists():
    prs = pd.read_parquet(LIGHT_PATH)
else:
    pr_path = hf_hub_download(HF_REPO, "all_pull_request.parquet", repo_type="dataset", revision=HF_REVISION)
    parts = []
    for batch in pq.ParquetFile(pr_path).iter_batches(batch_size=500_000, columns=PR_COLUMNS):
        df = batch.to_pandas()
        for col in ["created_at", "closed_at", "merged_at"]:
            df[col] = to_utc_naive(df[col])
        parts.append(df)
    prs = pd.concat(parts, ignore_index=True)
    for col in ["agent", "state", "user", "repo_url"]:
        prs[col] = prs[col].astype("category")
    prs.to_parquet(LIGHT_PATH, index=False)

prs["month"] = prs["created_at"].dt.to_period("M")
prs.info(memory_usage="deep")

In [ ]:
print(f"PRs:     {len(prs):>10,}")
print(f"Repos:   {prs['repo_url'].nunique():>10,}")
print(f"Authors: {prs['user'].nunique():>10,}")
print(f"Period:  {prs['created_at'].min():%Y-%m-%d} to {prs['created_at'].max():%Y-%m-%d}")

## Agentic PRs per month by agent

In [ ]:
monthly = prs.groupby(["month", "agent"], observed=True).size().unstack("agent", fill_value=0)

ax = monthly.plot(figsize=(10, 5), title="Agentic PRs per month by agent", rot=45)
ax.set_xlabel("")
ax.set_ylabel("PRs")
plt.tight_layout()
plt.savefig(FIG_DIR / "prs_per_month_by_agent.png", dpi=150)
plt.show()

monthly.tail(1).T.sort_values(monthly.index[-1], ascending=False)

## Repos by month of first agentic PR

In [ ]:
first_month = prs.groupby("repo_url", observed=True)["month"].min()
first_counts = first_month.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(first_counts.index.astype(str), first_counts.values)
ax.set_title("Repos by month of first agentic PR")
ax.set_ylabel("Repos")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIG_DIR / "repos_by_first_agentic_pr_month.png", dpi=150)
plt.show()

first_counts.loc[[pd.Period("2025-05", "M"), first_counts.index.max()]]

## Hours to merge and hours to close per agent

* **Hours to merge**: `merged_at - created_at` for merged PRs.
* **Hours to close**: `closed_at - created_at` for PRs closed *without* merging (same definition as the
  dataset's `scripts/productivity.py`).

Boxes span the IQR, whiskers p10–p90, outliers are hidden; the y axis is symlog (linear below 1 h).

In [ ]:
is_merged = prs["merged_at"].notna()
is_closed = prs["closed_at"].notna() & ~is_merged
prs["hours_to_merge"] = (prs["merged_at"] - prs["created_at"]).dt.total_seconds() / 3600
prs["hours_to_close"] = (prs["closed_at"] - prs["created_at"]).dt.total_seconds() / 3600

agents = sorted(prs["agent"].cat.categories)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, (col, mask, label) in zip(axes, [("hours_to_merge", is_merged, "Hours to merge"),
                                          ("hours_to_close", is_closed, "Hours to close")]):
    data = [prs.loc[mask & (prs["agent"] == a), col].dropna() for a in agents]
    ax.boxplot(data, tick_labels=agents, whis=(10, 90), showfliers=False)
    ax.set_yscale("symlog", linthresh=1)
    ax.set_title(f"{label} (box = IQR, whiskers = p10-p90)")
    ax.set_ylabel("Hours (symlog)")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIG_DIR / "hours_to_merge_and_close.png", dpi=150)
plt.show()

## Headline numbers on the slide

In [ ]:
# Concentration: share of PRs by the top 1% of authors
per_user = prs["user"].value_counts()
top1 = per_user.head(max(1, int(len(per_user) * 0.01))).sum() / per_user.sum()

# Breadth: share of repos that only ever used one agent
agents_per_repo = prs.groupby("repo_url", observed=True)["agent"].nunique()

print(f"Top 1% of users produce {top1:.1%} of PRs")
print(f"Repos using a single agent: {(agents_per_repo == 1).mean():.1%}")

pd.DataFrame({
    "PRs": prs.groupby("agent", observed=True).size(),
    "merge rate (merged / resolved)": is_merged.groupby(prs["agent"], observed=True).sum()
        / (is_merged | is_closed).groupby(prs["agent"], observed=True).sum(),
    "median hours to merge": prs.loc[is_merged].groupby("agent", observed=True)["hours_to_merge"].median(),
    "median minutes to merge": prs.loc[is_merged].groupby("agent", observed=True)["hours_to_merge"].median() * 60,
}).round(3)